# 04 — Grad-CAM Explainability

This notebook generates and analyses Grad-CAM heatmaps for the trained ResNet-50.

- Single image heatmap generation
- Batch heatmaps across all DR grades
- Failure case analysis (mis-graded images)
- Clinical interpretation of activation regions

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2

from src.utils.config import load_config, get_device
from src.models import build_resnet50
from src.data import get_val_transforms
from src.data.dataset import create_data_loaders
from src.explainability.gradcam import GradCAMVisualizer

cfg    = load_config('../configs/default.yaml')
device = get_device()

# Load trained model
model = build_resnet50(num_classes=5, fc_hidden=512, dropout=0.3,
                        pretrained=False).to(device)
ckpt = torch.load('../outputs/models/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Model loaded — best epoch: {ckpt["epoch"]} | Val QWK: {ckpt["val_qwk"]:.4f}')

## 1. Single Image Grad-CAM

In [ ]:
TRAIN_IMG = Path('../data/raw/aptos2019/train_images/train_images')
import pandas as pd
train = pd.read_csv('../data/raw/aptos2019/train_split.csv')

# Pick a Grade 3 (Severe) example
row = train[train['diagnosis'] == 3].iloc[0]
img_path = TRAIN_IMG / f"{row['id_code']}.png"
img_bgr = cv2.imread(str(img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

transform = get_val_transforms(224)
augmented = transform(image=img_rgb)
tensor = augmented['image'].unsqueeze(0).to(device)

cam = GradCAMVisualizer(model, device)
heatmap = cam.explainer.generate_heatmap(tensor)
heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
cam_overlay = cam.explainer.overlay_heatmap(img_rgb, heatmap_resized)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.resize(img_rgb, (224, 224)))
axes[0].set_title('Original Image', fontweight='bold')
axes[0].axis('off')
axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title('Grad-CAM Heatmap', fontweight='bold')
axes[1].axis('off')
axes[2].imshow(cam_overlay)
axes[2].set_title('Overlay (Grade 3 — Severe DR)', fontweight='bold')
axes[2].axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/gradcam_single.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Batch Grad-CAM Across All Grades

In [ ]:
_, _, test_loader = create_data_loaders(
    train_csv=cfg['paths']['train_csv'],
    val_csv=cfg['paths']['val_csv'],
    test_csv=cfg['paths']['test_csv'],
    train_image_dir=cfg['paths']['train_images'],
    val_image_dir=cfg['paths']['val_images'],
    test_image_dir=cfg['paths']['test_images'],
    train_transform=get_val_transforms(224),
    val_transform=get_val_transforms(224),
    image_size=224, batch_size=32,
)

cam.batch_generate(
    test_loader,
    n_samples=cfg['explainability']['num_samples'],
    output_dir='../outputs/gradcam/',
)
print('Batch Grad-CAM generation complete!')

## 3. Display Saved Heatmaps

In [ ]:
gradcam_dir = Path('../outputs/gradcam')
heatmap_files = sorted(gradcam_dir.glob('*.png'))[:10]

n = len(heatmap_files)
if n > 0:
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    for ax, fp in zip(axes.flat, heatmap_files):
        ax.imshow(cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB))
        ax.set_title(fp.stem[:25], fontsize=7)
        ax.axis('off')
    plt.suptitle('Grad-CAM Heatmaps', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No heatmaps yet — run Step 2 above')

## Clinical Interpretation

| DR Grade | Expected Activated Regions |
|---|---|
| 0 — No DR | Background / disc area (model uncertain) |
| 1 — Mild | Near optic disc: microaneurysms |
| 2 — Moderate | Perifoveal hemorrhages, hard exudates |
| 3 — Severe | Multiple quadrants: haemorrhages, IRMA |
| 4 — Proliferative | Neovascularisation, vitreous haemorrhage |

> 🔴 Red/yellow regions = high Grad-CAM activation (important for prediction)  
> The XAI layer enables **clinical trust** — clinicians can verify that the model attends to genuine lesion regions rather than artefacts.